<a href="https://colab.research.google.com/github/ingana1220/Diplomatura/blob/main/TP_Entrega_1_AM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification, make_regression, make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from xgboost import XGBRegressor
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
import pandas as pd
import geopandas as gpd



# Estilo visual limpio
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Todo listo. ")



✅ Todo listo. 


In [20]:
import requests

# URL del archivo "Raw" (crudo) de GitHub
url = "https://github.com/ingana1220/Diplomatura/blob/main/productos_belleza_skincare_v4.csv"

respuesta = requests.get(url)

# Verifica si la descarga fue exitosa
if respuesta.status_code == 200:
    contenido = respuesta.text
    print("ok") #contenido)
else:
    print("Error al cargar el archivo")

ok


In [21]:
import pandas as pd

# Raw GitHub URL for the CSV file
csv_url = 'https://raw.githubusercontent.com/ingana1220/Diplomatura/main/productos_belleza_skincare_v4.csv'

df = pd.read_csv(csv_url, sep='\t', encoding='latin-1')

# Clean column names: strip whitespace and replace spaces with underscores
df.columns = df.columns.str.strip().str.replace(' ', '_')

print(f"✅ Total de productos cargados: {len(df)}")

✅ Total de productos cargados: 267


In [23]:
# PASO2 - clasificación por palabras clave
# ============================================================
def clasificar_categoria(nombre):
    """
    Clasifica un producto de skincare en una de las 5 categorías
    según palabras clave presentes en su nombre.

    El orden de evaluación es importante:
    1. Protector solar (más específico)
    2. Limpiador
    3. Sérum
    4. Contorno de ojos
    5. Cremas (categoría más general, actúa como "catch-all")
    """
    nombre_lower = str(nombre).lower()

    # 1. PROTECTOR SOLAR (prioridad máxima)
    palabras_solar = ['spf', 'sunscreen', 'solar', 'fotoprotector', 'anthelios',
                      'fps', 'sun protection', 'protetor solar', 'heliocare',
                      'minesol', 'episol', 'biore uv', 'ultraceuticals ultra uv',
                      'cancer council sun', 'banana boat sun']
    if any(p in nombre_lower for p in palabras_solar):
        return 'Protector solar'

    # 2. LIMPIADORES (agua micelar, geles de limpieza)
    palabras_limpiador = ['cleanser', 'cleansing', 'gel de limpeza',
                          'micellar', 'micellaire', 'face wash', 'agua micelar',
                          'eau micellaire', 'micellar water', 'effaclar gel',
                          'clean & clear', 'demaquillante', 'daily facial cleanser',
                          'sensibio h2o', 'h2o micellaire']
    if any(p in nombre_lower for p in palabras_limpiador):
        return 'Limpiador'

    # 3. SÉRUM (tratamientos concentrados)
    palabras_serum = ['sérum', 'serum', 'advanced night repair',
                      'hyaluronic acid', 'hialuronico', 'hialurónico',
                      'vitamin c', 'retinol', 'green tea seed serum',
                      'hydra genius', 'mineral 89', 'rapid wrinkle repair',
                      'germinal', 'acglicolic', 'martiderm proteos',
                      'cellage firming', 'ordinary ha ', 'revitalift laser']
    if any(p in nombre_lower for p in palabras_serum):
        return 'Sérum'

    # 4. CONTORNO DE OJOS
    palabras_ojos = ['eye', 'contorno']
    if any(p in nombre_lower for p in palabras_ojos):
        return 'Contorno de ojos'

    # 5. CREMAS / HIDRATANTES (categoría general)
    palabras_crema = ['cream', 'crema', 'crème', 'lotion', 'loción', 'locion',
                      'loção', 'hidratante', 'moisturiser', 'moisturizing',
                      'moisturising', 'gel', 'baume', 'balm', 'body butter',
                      'cold cream', 'skin food', 'facial', 'máscara', 'mask',
                      'at4', 'moisture surge', 'liftactiv', 'hydractiv',
                      'babysquam', 'lata azul', 'boite bleue', 'oleo corporal',
                      'lipossolúvel', 'aqualia', 'toleriane fluide', 'aquaphor',
                      'ointment', 'pomada', 'spring water', 'eau thermale',
                      'corporal', 'reafirmante', 'antiage', 'antiedad']
    if any(p in nombre_lower for p in palabras_crema):
        return 'Cremas'

    return 'Otro / Sin clasificar'

# ============================================================
# PASO 3: Aplicar la clasificación
# ============================================================
print("DEBUG: Columnas del DataFrame antes de la clasificación:", df.columns)
df['categoria'] = df['nombre_producto'].apply(clasificar_categoria)

# ============================================================
# PASO 4: Resultados y guardado
# ============================================================
print("\n📊 DISTRIBUCIÓN DE CATEGORÍAS:")
print(df['categoria'].value_counts())

print("\n🌍 TABLA CRUZADA: PAÍS vs CATEGORÍA")
print(pd.crosstab(df['pais'], df['categoria']))

# Guardar el CSV enriquecido
df.to_csv('productos_belleza_con_categoria.csv', sep='\t', index=False)
print(f"\n💾 Archivo guardado con {len(df)} filas y {len(df.columns)} columnas")

DEBUG: Columnas del DataFrame antes de la clasificación: Index(['nombre_producto', 'unidad_medida', 'cantidad', 'precio', 'moneda',
       'pais', 'cotizacion_USD', 'Valor_prod_USD', 'USD_x_cant', 'categoria'],
      dtype='object')

📊 DISTRIBUCIÓN DE CATEGORÍAS:
categoria
Cremas                   179
Sérum                     42
Limpiador                 22
Protector solar           17
Otro / Sin clasificar      7
Name: count, dtype: int64

🌍 TABLA CRUZADA: PAÍS vs CATEGORÍA
categoria  Cremas  Limpiador  Otro / Sin clasificar  Protector solar  Sérum
pais                                                                       
Argentina      39          5                      1                2      7
Australia      35          6                      0                6     10
Brasil         33          5                      4                4      8
España         40          3                      2                3     10
Francia        32          3                      0            

In [24]:
def corregir_codificacion(archivo):
    """
    Corrige problemas de codificación en un archivo:
    - SÃ©rum → Sérum
    - EspaÃ±a → España
    """
    # Leer el archivo con la codificación correcta
    with open(archivo, 'r', encoding='latin-1') as f:
        contenido = f.read()

    # Reemplazar los caracteres con problemas de codificación
    contenido = contenido.replace('SÃ©rum', 'Sérum')
    contenido = contenido.replace('EspaÃ±a', 'España')

    # Escribir el archivo corregido con la codificación correcta
    with open(archivo, 'w', encoding='latin-1') as f:
        f.write(contenido)

    print(f"Archivo '{archivo}' corregido exitosamente")

# Uso: Apuntamos al archivo CSV generado anteriormente
archivo_a_corregir = '/content/productos_belleza_con_categoria.csv'
corregir_codificacion(archivo_a_corregir)

Archivo '/content/productos_belleza_con_categoria.csv' corregido exitosamente


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import json
import streamlit as st
import pandas as pd
from pathlib import Path

st.set_page_config(page_title="Análisis Belleza", layout="wide")

@st.cache_data
def cargar_datos(ruta: str) -> pd.DataFrame:
    # Ensure correct encoding and separator are used when loading the file
    df = pd.read_csv(ruta, encoding='latin-1', sep='\t')

    # Clean column names: strip whitespace and replace spaces with underscores (for robustness)
    df.columns = df.columns.str.strip().str.replace(' ', '_')

    # Convert numeric columns from string (with comma decimal) to float
    # This handles values like '15,9' becoming 15.9
    for col in ['precio', 'Valor_prod_USD', 'USD_x_cant', 'cotizacion_USD', 'cantidad']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.replace(',', '.').astype(float)

    # Create 'precio_ml' from 'USD_x_cant' as it represents 'USD per quantity'
    df['precio_ml'] = df['USD_x_cant']

    # Extract 'marca' from 'nombre_producto' (assuming first word is the brand)
    # Handle potential non-string values in 'nombre_producto' by converting to string first
    df['marca'] = df['nombre_producto'].astype(str).apply(lambda x: x.split(' ')[0] if x else 'Desconocida')

    return df

def preparar_payload(df: pd.DataFrame):
    """Convierte el DataFrame en el JSON que espera el HTML."""
    agg = df.groupby(['pais', 'categoria']).agg(
        # Use 'Valor_prod_USD' for price aggregation for consistency with USD plots
        precio=('Valor_prod_USD', 'mean'),
        unit=('precio_ml', 'mean'),
        count=('nombre_producto', 'size') # Using 'nombre_producto' for count as it's less ambiguous
    ).reset_index()

    data = {}
    for pais in agg['pais'].unique():
        data[pais] = {}
        sub = agg[agg['pais'] == pais]
        for _, row in sub.iterrows():
            data[pais][row['categoria']] = {
                'precio': round(row['precio'], 2),
                'unit': round(row['unit'], 2),
                'count': int(row['count'])
            }

    marcas_agg = df.groupby('marca').agg(
        precioEtiqueta=('Valor_prod_USD', 'mean'), # Use 'Valor_prod_USD' here too
        precioMl=('precio_ml', 'mean')
    ).reset_index()
    marcas = [
        {
            'marca': r['marca'],
            'precioEtiqueta': round(r['precioEtiqueta'], 2),
            'precioMl': round(r['precioMl'], 2),
        }
        for _, r in marcas_agg.iterrows()
    ]

    return {
        'data': data,
        'categorias': sorted(df['categoria'].unique().tolist()),
        'paises': sorted(df['pais'].unique().tolist()),
        'marcas': marcas,
    }

# ---------- UI ----------
st.title("🧴 Análisis de Precios - Productos de Belleza")

df = cargar_datos('/content/productos_belleza_con_categoria.csv')

# Filtros interactivos (esto es lo que Streamlit aporta sobre HTML puro)
col1, col2 = st.columns(2)
with col1:
    paises_sel = st.multiselect("Países", sorted(df['pais'].unique()),
                                default=sorted(df['pais'].unique()))
with col2:
    cats_sel = st.multiselect("Categorías", sorted(df['categoria'].unique()),
                              default=sorted(df['categoria'].unique()))

df_filtrado = df[df['pais'].isin(paises_sel) & df['categoria'].isin(cats_sel)]
payload = preparar_payload(df_filtrado)

# Renderizar el HTML con los datos inyectados
# NOTE: Corrected path for dashboard.html
html_path = Path('/content/sample_data/Dashboard Beauty TP.html')
html_content = html_path.read_text(encoding='utf-8')

# Inyectar datos reemplazando los placeholders Jinja manualmente
# (alternativa simple sin Jinja2)
html_inyectado = html_content.replace(
    "{{ data_json | safe }}", json.dumps(payload['data'])
).replace(
    "{{ categorias | safe }}", json.dumps(payload['categorias'])
).replace(
    "{{ paises | safe }}", json.dumps(payload['paises'])
).replace(
    "{{ marcas | safe }}", json.dumps(payload['marcas'])
)

# Revert to st.components.v1.html as st.iframe does not support srcdoc
st.components.v1.html(html_inyectado, height=2400, scrolling=True)


2026-07-04 03:42:46.502 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-04 03:42:46.506 No runtime found, using MemoryCacheStorageManager
2026-07-04 03:42:46.513 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-04 03:42:46.514 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-04 03:42:46.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-04 03:42:46.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-04 03:42:46.522 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-04 03:42:46.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-04 03:42:46.525 Thread 'MainThread':

DeltaGenerator()